Bert 전이학습

In [58]:
from tensorflow.keras.utils import get_file

ratings_train_path = get_file(
    "ratings_train.txt",
    "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"
)

ratings_test_path = get_file(
    "ratings_test.txt",
    "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt"
)

print(ratings_train_path)
print(ratings_test_path)

C:\Users\SJ\.keras\datasets\ratings_train.txt
C:\Users\SJ\.keras\datasets\ratings_test.txt


In [59]:
import pandas as pd

# 데이터 (tsv) 로드 -> DataFrame으로 변환
ratings_train_df = pd.read_csv(ratings_train_path, sep='\t')
ratings_test_df = pd.read_csv(ratings_test_path, sep='\t')

display(ratings_train_df.head())
display(ratings_test_df.head())

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


,id,document,label
0,6270596,굳 ㅋ,1
1,9274899,GDNTOPCLASSINTHECLUB,0
2,8544678,뭐야 이 평점들은.... 나쁘진 않지만 10점 짜리는 더더욱 아니잖아,0
3,6825595,지루하지는 않은데 완전 막장임... 돈주고 보기에는....,0
4,6723715,3D만 아니었어도 별 다섯 개 줬을텐데.. 왜 3D로 나와서 제 심기를 불편하게 하죠??,0


In [60]:
ratings_train_df.isnull().sum(), ratings_test_df.isnull().sum()

(id          0
 document    5
 label       0
 dtype: int64,
 id          0
 document    3
 label       0
 dtype: int64)

In [61]:
print(ratings_train_df.info())
print(ratings_test_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   id        150000 non-null  int64
 1   document  149995 non-null  str  
 2   label     150000 non-null  int64
dtypes: int64(2), str(1)
memory usage: 3.4 MB
None
<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        50000 non-null  int64
 1   document  49997 non-null  str  
 2   label     50000 non-null  int64
dtypes: int64(2), str(1)
memory usage: 1.1 MB
None


In [62]:
# CPU에서도 실습 가능한 크기로 샘플링
ratings_train_df = ratings_train_df.sample(n=3_000, random_state=0)
ratings_test_df = ratings_test_df.sample(n=1_000, random_state=0)

ratings_train_df['label'].value_counts(), ratings_test_df['label'].value_counts()

(label
 1    1517
 0    1483
 Name: count, dtype: int64,
 label
 1    508
 0    492
 Name: count, dtype: int64)

In [63]:
# 텍스트/라벨을 리스트로 변환
X_train = ratings_train_df['document'].fillna('').values.tolist()
y_train = ratings_train_df['label'].values.tolist()

X_test = ratings_test_df['document'].fillna('').values.tolist()
y_test = ratings_test_df['label'].values.tolist()


토크나이저/모델 준비
- bert 한국어 버전 사전학습 모델 klue/bert-base

In [64]:
from transformers import AutoTokenizer

model_name = 'klue/bert-base'
tokenizer = AutoTokenizer.from_pretrained(model_name) # 토크나이저 (토큰화 규칙/어휘사전)

In [65]:
# 데이터 문장들 토큰화 + 패딩/자르기 처리 + Tensor 변환
X_train = tokenizer(X_train, padding=True, truncation=True, max_length=64)
X_test = tokenizer(X_test, padding=True, truncation=True, max_length=64)

X_train[:3], X_test[:3]

([Encoding(num_tokens=64, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
  Encoding(num_tokens=64, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
  Encoding(num_tokens=64, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])],
 [Encoding(num_tokens=64, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
  Encoding(num_tokens=64, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
  Encoding(num_tokens=64, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])])

In [66]:
print(X_train['input_ids'][0])       # 토큰이 정수 ID로 변환된 시퀀스
print(X_train['attention_mask'][0])  # 실제 토큰=1, 패딩=0 으로 구분한 마스크 
print(X_train['token_type_ids'][0])  # 문장 구분 ID 

[2, 7180, 2137, 31369, 2119, 3897, 2776, 2332, 809, 3897, 2223, 2259, 7658, 2530, 18, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


데이터 파이프라인 생성

In [67]:
import torch
from torch.utils.data import Dataset, DataLoader

# BERT 입력 데이터와 라벨을 함께 관리하는 Dataset 클래스
class NSMCDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings  # 토큰화 결과(input_ids, attention_mask 등) 저장
        self.labels = labels        # 정답 라벨

    def __len__(self):
        return len(self.labels)

    # idx 번째 샘플의 토큰화 결과(input_ids, attention_mask 등)을 딕셔너리 형태로 반환
    def __getitem__(self, idx):
        item = {
            key: torch.tensor(value[idx], dtype=torch.long)
            for key, value in self.encodings.items()
        }

        # idx번째 정답 라벨을 Pytorch 정수형 LongTensor 형태로 변환해서 추가
        item['labels'] = torch.tensor(self.labels[idx], dtype = torch.long)

        return item

In [68]:
# Pytorch Dataset 생성
train_dataset = NSMCDataset(X_train, y_train)
test_dataset = NSMCDataset(X_test, y_test)

# CPU 메모리와 처리 속도를 고려해 16개씩 배치
train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False)


In [69]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(model_name, num_labels = 2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [70]:
# 파인튜닝 설정
import torch
from transformers import get_scheduler # 학습률 스케줄러 생성 함수

epochs = 2

# AdamW 최적화함수 : 가중치 감쇠 적용(과적합 완화)
optimizer = torch.optim.AdamW(model.parameters(), lr =5e-5, weight_decay = 0.1)
  
num_train_steps = len(train_dataloader) * epochs # 전체 학습 step수

# warmup step 수 : 전체 학습 step에서 10%는 학습률을 점진적으로 증가
# - 사전학습 fine-tuning시에는 초반에 LR가 크면 학습이 불안정해지는 경우가 많아서 0에서부터 lr까지 점차적으로 학습률을 늘려준다.
num_warmup_steps = int(num_train_steps * 0.1)

lr_scheduler = get_scheduler(
    name = 'linear',        # Warmup 이후 학습률 선형적으로 감소
    optimizer = optimizer,
    num_warmup_steps = num_warmup_steps,
    num_training_steps = num_train_steps
)

In [71]:
from tqdm.auto import tqdm 

# 학습
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

for epoch in tqdm(range(epochs)):
    model.train()
    total_loss = 0

    for batch in tqdm(train_dataloader, desc=f'Epoch {epoch + 1}/{epochs}'):
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        lr_scheduler.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_dataloader)
    print(f'Epoch {epoch + 1}/{epochs} - 평균 손실: {avg_loss:.4f}')

  0%|          | 0/2 [00:00<?, ?it/s]

Epoch 1/2:   0%|          | 0/188 [00:00<?, ?it/s]

Epoch 1/2 - 평균 손실: 0.4444


Epoch 2/2:   0%|          | 0/188 [00:00<?, ?it/s]

Epoch 2/2 - 평균 손실: 0.1831


In [72]:
model.save_pretrained('nsmc_model/bert-base') # 학습된 모델 저장
tokenizer.save_pretrained('nsmc_model/bert-base') # 모델에서 사용한 토크나이저 지정

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('nsmc_model/bert-base\\tokenizer_config.json',
 'nsmc_model/bert-base\\tokenizer.json')

In [73]:
# 감성분석 파이프라인 생성
from transformers import TextClassificationPipeline

# 입력 텍스트 -> 토큰화 -> 모델 추론 -> 라벨/점수 반환 파이프라인
sentiment_classifier = TextClassificationPipeline(
    tokenizer = tokenizer,
    model = model,
    framework = 'pt',     # PyTorch 기반 모델
    top_k = None          # 모든 클래스 라벨과 확률 반환
)

### HuggingFace 업로드

In [ ]:
from huggingface_hub import notebook_login

# Write 권한 토큰 입력 후 Login 버튼을 누릅니다.
# 인증이 끝난 다음에 반드시 아래 업로드 셀을 실행합니다.
notebook_login()

In [90]:
# HuggingFace에 학습한 모델 업로드
from huggingface_hub import get_token, whoami

HF_TOKEN = get_token()
if not HF_TOKEN:
    raise RuntimeError("로그인 셀에서 토큰 입력 후 Login 버튼을 먼저 누르세요.")

HF_USERNAME = whoami(token=HF_TOKEN)['name']
REPO_ID = f'SONG/bert-base-nsmc'
print(f'업로드 대상: {REPO_ID}')

model.push_to_hub(REPO_ID, token=HF_TOKEN)
tokenizer.push_to_hub(REPO_ID, token=HF_TOKEN)

RuntimeError: 로그인 셀에서 토큰 입력 후 Login 버튼을 먼저 누르세요.

In [91]:
# Hugging Face에 업로드한 내 모델을 로드해서 가져옴
from transformers import AutoTokenizer, AutoModelForSequenceClassification
HUB_NAME = REPO_ID

tokenizer = AutoTokenizer.from_pretrained(HUB_NAME)
model = AutoModelForSequenceClassification.from_pretrained(HUB_NAME)

NameError: name 'REPO_ID' is not defined

In [92]:
# 감정분류기 생성
from transformers import pipeline

sentiment_classifier = pipeline(
    'text-classification',
    model=HUB_NAME,
    tokenizer=HUB_NAME,
    framework='pt',
    top_k=None
)

NameError: name 'HUB_NAME' is not defined

In [ ]:
sentiment_classifier([
    '한국 영화는 이래서 안돼~',
    '역시 봉감독이 최고야!',
    '진짜 정말로 강하게 재미없다.',
    
])